# CSE — Price Prediction Baseline

Baseline models for next-day return prediction:
1. Naive benchmark (predict mean / zero return)
2. Linear regression on lagged features
3. Random Forest classifier (direction: up/down)

**Target:** `return_1d` (next-day return) for a single liquid stock.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report
)
from sklearn.pipeline import Pipeline

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
df = pd.read_parquet('../data/published/cse_unified.parquet')
df['date'] = pd.to_datetime(df['date'])

# Pick the most liquid stock (highest avg turnover)
top_symbol = (
    df[df['is_trading_day']]
    .groupby('symbol')['turnover']
    .mean()
    .idxmax()
)
print(f'Selected symbol: {top_symbol}')

stock = df[df['symbol'] == top_symbol].copy().sort_values('date').reset_index(drop=True)
print(f'Rows: {len(stock)}, date range: {stock["date"].min().date()} to {stock["date"].max().date()}')

## 1. Feature Construction

In [ ]:
# Lag features
for lag in [1, 2, 3, 5]:
    stock[f'ret_lag{lag}'] = stock['return_1d'].shift(lag)

stock['vol_lag1']   = stock['volatility_20d'].shift(1)
stock['ma50_lag1']  = stock['close_to_ma50'].shift(1)
stock['ma200_lag1'] = stock['close_to_ma200'].shift(1)
stock['vol_z_lag1'] = stock['volume_zscore'].shift(1)

# Target: next-day return
stock['target_ret']       = stock['return_1d']
stock['target_direction'] = (stock['return_1d'] > 0).astype(int)

FEATURES = ['ret_lag1','ret_lag2','ret_lag3','ret_lag5',
            'vol_lag1','ma50_lag1','ma200_lag1','vol_z_lag1']

clean = stock[FEATURES + ['target_ret','target_direction','date']].dropna()
print(f'Clean rows for modelling: {len(clean)}')

## 2. Walk-Forward Train / Test Split (80 / 20 chronological)

In [ ]:
split = int(len(clean) * 0.80)
train = clean.iloc[:split]
test  = clean.iloc[split:]

X_train = train[FEATURES].values
y_train_ret = train['target_ret'].values
y_train_dir = train['target_direction'].values

X_test = test[FEATURES].values
y_test_ret = test['target_ret'].values
y_test_dir = test['target_direction'].values

print(f'Train: {len(train)} rows ({train["date"].min().date()} to {train["date"].max().date()})')
print(f'Test:  {len(test)} rows ({test["date"].min().date()} to {test["date"].max().date()})')

## 3. Naive Benchmark

In [ ]:
naive_pred_ret = np.full(len(y_test_ret), y_train_ret.mean())
naive_pred_dir = np.full(len(y_test_dir), int(y_train_dir.mean() >= 0.5))

naive_rmse = np.sqrt(mean_squared_error(y_test_ret, naive_pred_ret))
naive_acc  = accuracy_score(y_test_dir, naive_pred_dir)

print(f'Naive benchmark — RMSE: {naive_rmse:.5f}  |  Direction accuracy: {naive_acc:.3f}')

## 4. Ridge Regression (return prediction)

In [ ]:
ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))])
ridge.fit(X_train, y_train_ret)
ridge_pred = ridge.predict(X_test)

ridge_rmse = np.sqrt(mean_squared_error(y_test_ret, ridge_pred))
ridge_mae  = mean_absolute_error(y_test_ret, ridge_pred)
ridge_acc  = accuracy_score(y_test_dir, (ridge_pred > 0).astype(int))

print(f'Ridge — RMSE: {ridge_rmse:.5f}  MAE: {ridge_mae:.5f}  Direction acc: {ridge_acc:.3f}')

print('\nFeature coefficients:')
coefs = pd.Series(ridge.named_steps['model'].coef_, index=FEATURES).sort_values()
print(coefs.round(5).to_string())

## 5. Random Forest Classifier (direction prediction)

In [ ]:
rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=200, max_depth=5,
                                      min_samples_leaf=20, random_state=42))
])
rf.fit(X_train, y_train_dir)
rf_pred = rf.predict(X_test)
rf_acc  = accuracy_score(y_test_dir, rf_pred)

print(f'Random Forest direction accuracy: {rf_acc:.3f}')
print()
print(classification_report(y_test_dir, rf_pred, labels=[0, 1], target_names=['Down','Up'], zero_division=0))

importances = pd.Series(
    rf.named_steps['model'].feature_importances_, index=FEATURES
).sort_values(ascending=False)
print('Feature importances:')
print(importances.round(4).to_string())

## 6. Predictions vs Actuals (test period)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(test['date'].values, y_test_ret, lw=0.8, label='Actual', alpha=0.8)
axes[0].plot(test['date'].values, ridge_pred, lw=0.8, label='Ridge pred', alpha=0.8)
axes[0].set_ylabel('Daily return')
axes[0].set_title(f'{top_symbol} — Ridge Return Prediction vs Actual (test set)')
axes[0].legend()

# Cumulative return: buy-and-hold vs model-directed
bh = (1 + pd.Series(y_test_ret)).cumprod()
model_signal = np.where(ridge_pred > 0, 1, -1)
model_returns = model_signal * y_test_ret
model_cum = (1 + pd.Series(model_returns)).cumprod()

axes[1].plot(test['date'].values, bh.values, lw=1.2, label='Buy & Hold')
axes[1].plot(test['date'].values, model_cum.values, lw=1.2, label='Ridge-directed')
axes[1].set_ylabel('Cumulative return')
axes[1].set_title('Cumulative Return: Buy & Hold vs Ridge-Directed Strategy')
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nSummary:')
print(f'  Naive RMSE:  {naive_rmse:.5f}  |  acc: {naive_acc:.3f}')
print(f'  Ridge RMSE:  {ridge_rmse:.5f}  |  acc: {ridge_acc:.3f}')
print(f'  RF acc:      {rf_acc:.3f}')